In [97]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt.tool_node import ToolNode, tools_condition
from langgraph.graph.message import add_messages

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_groq.chat_models import ChatGroq

from langchain_huggingface import HuggingFaceEmbeddings

from langsmith import traceable

from dotenv import load_dotenv
from typing import Annotated, TypedDict



import os

load_dotenv()

True

In [83]:
# llm
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")

HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")
HUGGINGFACE_AMBEDDING_MODEL=os.getenv("HUGGINGFACE_AMBEDDING_MODEL")

embeddings = HuggingFaceEmbeddings(model=HUGGINGFACE_AMBEDDING_MODEL)

if not GROQ_API_KEY or not GROQ_MODEL:
    raise RuntimeError("GROQ_API_KEY / GROQ_MODEL missing from env")

llm = ChatGroq(model=GROQ_MODEL, api_key=GROQ_API_KEY)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 913.65it/s]


In [84]:
loader = PyPDFLoader("intro-to-ai.pdf")
docs = loader.load()

In [85]:
len(docs)

44

In [86]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

chunks = splitter.split_documents(docs)

In [87]:
vector_stor = FAISS.from_documents(chunks, embeddings)

In [88]:
retriever = vector_stor.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [89]:
result = retriever.invoke("What is Turning Machine?")

print(result)

[Document(id='33d136a5-2824-4341-b128-ada83c0b6826', metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.0 (Macintosh)', 'creationdate': '2024-03-06T16:39:02+00:00', 'moddate': '2024-03-06T16:39:08+00:00', 'trapped': '/False', 'source': 'intro-to-ai.pdf', 'total_pages': 44, 'page': 7, 'page_label': '8'}, page_content='INTRODUCTION TO AI\nWorld Travel & Tourism Council\n8\n< Contents  |\nSome AI technologies have been around for more than 50 years, but advances in computing power, the \navailability of enormous quantities of data and new developments in software algorithms have led to major \nAI breakthroughs in recent years. It is these three components of advanced algorithms, data and computing \npower, that explain how machines can exhibit intelligent behaviour and why AI has suddenly exploded into our \neveryday lives. \nAlgorithms tell computers what to do. Data tells computers what to learn. \nComputing power gives machines the power to learn and make dec

In [ ]:
@tool
def rag_tool(query):
    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents.
    """
    
    result = retriever.invoke(query)
    
    content = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    
    return {
        "query": query,
        "content": content,
        "metadata": metadata
    }

In [91]:
tools = [rag_tool]

llm_with_tools = llm.bind_tools(tools)

In [92]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [93]:
def chat_node(state: ChatState):
    messages = state["messages"]
    
    response = llm_with_tools.invoke(messages)
    
    return {
        "messages": [response]
    }

In [94]:
tool_node = ToolNode(tools)

In [95]:
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge("tools", "chat_node")

chatbot = graph.compile()

In [96]:
result = chatbot.invoke({"messages": [HumanMessage(content=("Who is the Julia Simpson"))]})

print(result["messages"][-1].content)

Julia Simpson is the President & Chief Executive Officer of the **World Travel & Tourism Council (WTTC)**. In the AI‑focused PDF you’re looking at, she contributed the foreword, signing it as:

> “Julia Simpson – President & CEO, World Travel & Tourism Council”

In that role, she leads the global organization that represents the travel and tourism industry, advocating for policies that support sustainable growth and the sector’s contribution to the world economy. Her foreword frames the discussion of AI’s impact on travel, tourism, and broader societal issues.
